# Reconstrução de imagens de RM com SigPy e Compressed Sensing

**Objetivo:** explorar o SigPy num fluxo simplificado de reconstrução de ressonância magnética.

Fluxo: imagem → espaço-$k$ → subamostragem → reconstrução (*zero-filled* vs $L_1$-wavelet) → métricas (PSNR/SSIM).

**Como usar este notebook:** cada figura do relatório tem a **sua própria célula**. No topo de cada célula há um bloco
`# >>> PARÂMETROS` — é só isso que precisas de mudar para obter uma variante da figura. As figuras aparecem aqui
e são também guardadas em `figures/` com o nome usado no relatório.

| Figura (ficheiro em `figures/`) | Secção | O que mudar |
|---|---|---|
| `exp0_zero_filled_R4.png` | 3 | `R`, `SEED` |
| `exp1_kspace_center_vs_periphery.png` | 2 | `RAIO` |
| `exp3_mask_type_R4.png` | 4 | `R`, `LAMBDA` |
| `metrics.csv` (tabela) | 5 | `RS`, `LAMBDA` |
| `exp2_acceleration.png` | 6 | `MASCARA` |
| `exp2_metrics_vs_R.png` | 7 | `MASCARAS` |
| `exp5_metrics_bars.png` | 8 | — |
| `exp4_lambda.png`, `exp4_lambda_curve.png` | 9 | `R`, `LAMBDAS` |
| `exp6_noise.png` | 10 | `R`, `SIGMAS` |

> **Nota:** a imagem MNI152 já é uma imagem reconstruída. O espaço-$k$ é simulado por transformada de Fourier;
> não são dados brutos de scanner nem dados de doentes.

## 0. Preparar o ambiente

A célula seguinte instala as bibliotecas e entra na pasta do projeto. Procura, por esta ordem:
1. o notebook já está **dentro** da pasta `mri-sigpy-demo` (caso normal no computador);
2. a pasta `mri-sigpy-demo` ou o ficheiro `mri-sigpy-demo.zip` estão **ao lado** do notebook;
3. no Colab, pede para carregar o ZIP.

In [ ]:
%pip install --upgrade pandas scipy scikit-image numba

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 36.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.3/35.3 MB 45.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.7/13.7 MB 121.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.9/3.9 MB 121.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.7/59.7 MB 61.9 MB/s  0:00:00
  Attempting uninstall: scipy
    Found existing installation: scipy 1.16.3
    Uninstalling scipy-1.16.3:
      Successfully uninstalled scipy-1.16.3
  Attempting uninstall: llvmlite
    Found existing installation: llvmlite 0.46.0
    Uninstalling llvmlite-0.46.0:
      Successfully uninstalled llvmlite-0.46.0
  Attempting uninstall: scikit-image
    Found existing installation: scikit-image 0.25.2
    Uninstalling scikit-image-0.25.2:
      Successfully uninstalled scikit-image-0.25.2
  Attempting uninstall: pandas
    Found existing installation: pandas 2.2.3
    Uninstalling pandas-2.2.3:
      Successfully 

In [ ]:
%pip install -q sigpy nibabel scikit-image PyWavelets pandas

import os, sys, zipfile
from pathlib import Path


def procurar_projeto():
    achados = sorted(Path(".").rglob("mrisim.py"))
    return achados[0].parent if achados else None


def entrar_no_projeto():
    pasta = Path(".") if Path("mrisim.py").exists() else procurar_projeto()

    if pasta is None:                       # ainda não temos o projeto: usar o ZIP
        zips = sorted(Path(".").glob("*.zip"))
        if not zips:                        # Colab: pedir o ZIP
            try:
                from google.colab import files
            except ImportError:
                raise FileNotFoundError(
                    "Não encontrei o projeto. Abre o notebook dentro da pasta que tem o mrisim.py "
                    f"ou põe o .zip ao lado do notebook. Pasta atual: {Path.cwd()}")
            print("Seleciona o ficheiro .zip do projeto")
            zips = [Path(n) for n in files.upload() if n.endswith(".zip")]
        for z in zips:
            zipfile.ZipFile(z).extractall(".")
        pasta = procurar_projeto()


print(entrar_no_projeto())
sys.path.insert(0, os.getcwd())
print("Pasta do projeto:", Path.cwd())

Seleciona o ficheiro .zip do projeto


Saving mri-sigpy-demo.zip to mri-sigpy-demo.zip
None
Pasta do projeto: /content


In [ ]:
import os
print("Pasta atual:", os.getcwd())
for raiz, _, fich in os.walk("/content"):
    if raiz.count("/") > 3 or "sample_data" in raiz:
        continue
    for f in fich:
        if f.endswith((".py", ".zip", ".gz")):
            print(os.path.join(raiz, f))
os.chdir("/content/mri-sigpy-demo")

Pasta atual: /content
/content/mri-sigpy-demo.zip
/content/mri-sigpy-demo/mrisim.py
/content/mri-sigpy-demo/run_demo.py
/content/mri-sigpy-demo/figures.py
/content/mri-sigpy-demo/data/mni152_t1.nii.gz


## 1. Bibliotecas e parâmetros globais

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import sigpy as sp

from mrisim import load_slice, to_kspace, add_noise, make_mask, recon_zf, recon_cs, metrics
import figures as F            # uma função por figura (ver figures.py)

# >>> PARÂMETROS GLOBAIS (valem para todo o notebook)
FATIA    = None      # índice da fatia axial do MNI152 (None = fatia do meio)
SEED     = 0         # semente da máscara aleatória (muda o sorteio das linhas)
LAMBDA   = 1e-2      # regularização L1-wavelet
MAX_ITER = 100       # iterações do algoritmo
SAVE     = "figures" # pasta onde as figuras são guardadas (None = não guardar)

print("SigPy:", getattr(sp, "__version__", "?"))

SigPy: 0.1.27


## 2. Imagem original e espaço-$k$ (Figura `exp1`)

O centro do espaço-$k$ (baixas frequências) contém o contraste e a forma global; a periferia (altas frequências)
contém os contornos e o detalhe fino.

**Para mudar a figura:** `RAIO` — quanto maior, mais frequências ficam no "centro" e mais nítida fica a reconstrução só com o centro.

In [ ]:
# >>> PARÂMETROS
RAIO = 20   # raio (em pixéis de k-space) que separa centro de periferia

img = load_slice(z=FATIA)
ksp = to_kspace(img)
print("Dimensões da imagem:", img.shape)

F.fig_kspace(img, ksp, radius=RAIO, save_dir=SAVE)
plt.show()

## 3. Subamostragem: máscara regular vs aleatória (Figura `exp0`)

$R$ = fator de aceleração ≈ fração $1/R$ das linhas de $k$ adquiridas. As amostras em falta ficam a zero
(*zero-filled*). A máscara regular cria réplicas (*aliasing* coerente); a aleatória espalha o erro como ruído
(*aliasing* incoerente), que é o que o compressed sensing consegue remover.

**Para mudar a figura:** `R` (2, 4, 8, ...) e `SEED` (global).

In [ ]:
# >>> PARÂMETROS
R = 4

F.fig_zero_filled(img, ksp, R=R, seed=SEED, save_dir=SAVE)
plt.show()

## 4. Reconstrução compressed sensing $L_1$-wavelet (Figura `exp3`)

`sigpy.mri.app.L1WaveletRecon` procura a imagem $x$ que minimiza
$\tfrac12\|M\mathcal{F}x - y\|_2^2 + \lambda\|\Psi x\|_1$ ($\Psi$ = transformada wavelet).
`mps` são os mapas de sensibilidade das bobinas; aqui é uma matriz de uns (uma bobina ideal).

Primeiro o passo-a-passo de uma reconstrução só, depois a figura completa regular vs aleatória.

In [ ]:
# >>> PARÂMETROS
R = 4

# Passo a passo (é isto que recon_cs faz por dentro)
import sigpy.mri as mr
mascara = make_mask("random", img.shape, R, seed=SEED)
k_sub   = ksp * mascara                                    # dados "adquiridos"
mps     = np.ones((1,) + k_sub.shape, dtype=np.complex64)  # 1 bobina, sensibilidade constante
x_cs    = np.abs(mr.app.L1WaveletRecon(k_sub[None], mps, LAMBDA,
                                        max_iter=MAX_ITER, show_pbar=False).run())
p, s = metrics(img, x_cs)
print(f"L1-wavelet (R={R}, λ={LAMBDA:g}): PSNR = {p:.2f} dB | SSIM = {s:.4f}")

In [ ]:
# Figura do relatório: máscara | zero-filled | L1-wavelet | erro, para as duas máscaras
F.fig_mask_type(img, ksp, R=R, lamda=LAMBDA, max_iter=MAX_ITER, seed=SEED, save_dir=SAVE)
plt.show()

## 5. Tabela de resultados para vários $R$ (`results/metrics.csv`)

Corre zero-filled e $L_1$-wavelet para todas as combinações de $R$ e tipo de máscara.
As secções 6, 7 e 8 usam estes resultados — **corre esta célula antes delas** (demora ~10–20 s em CPU).

In [ ]:
# >>> PARÂMETROS
RS      = (2, 4, 8)
MASCARAS_TABELA = ("regular", "random")

rows, store = F.run_experiments(img, ksp, Rs=RS, masks=MASCARAS_TABELA,
                                lamda=LAMBDA, max_iter=MAX_ITER, seed=SEED, verbose=False)
F.save_table(rows, "results/metrics.csv")
tabela = pd.DataFrame(rows)
tabela

In [ ]:
# Mesma tabela em formato "largo" (mais fácil de comparar ZF vs CS)
tabela.pivot_table(index=["mask", "R"], columns="method", values=["PSNR", "SSIM"])

## 6. Imagens para vários fatores de aceleração (Figura `exp2_acceleration`)

**Para mudar a figura:** `MASCARA` (`"random"` ou `"regular"`). Os valores de $R$ vêm de `RS` (secção 5).

In [ ]:
# >>> PARÂMETROS
MASCARA = "random"

F.fig_acceleration(img, store, Rs=RS, mask=MASCARA, save_dir=SAVE)
plt.show()

## 7. PSNR e SSIM em função de $R$ (Figura `exp2_metrics_vs_R`)

**Para mudar a figura:** `MASCARAS = ("random",)` reproduz a figura do relatório;
`("regular", "random")` mostra as duas máscaras no mesmo gráfico.

In [ ]:
# >>> PARÂMETROS
MASCARAS = ("random",)

F.fig_metrics_vs_R(rows, masks=MASCARAS, save_dir=SAVE)
plt.show()

## 8. Comparação em barras: máscara regular vs aleatória (Figura `exp5_metrics_bars`)

In [ ]:
F.fig_metrics_bars(rows, save_dir=SAVE)
plt.show()

## 9. Influência do parâmetro de regularização $\lambda$ (Figuras `exp4_lambda` e `exp4_lambda_curve`)

$\lambda$ pequeno → pouca regularização (fica parecido com o zero-filled);
$\lambda$ grande → imagem demasiado suavizada / com perda de detalhe. O gráfico mostra onde está o melhor compromisso.

**Para mudar a figura:** `R` e `LAMBDAS` (mais valores = curva mais suave, mas mais lento).

In [ ]:
# >>> PARÂMETROS
R       = 4
LAMBDAS = (1e-4, 1e-3, 1e-2, 1e-1, 1.0)

sweep = F.lambda_sweep(img, ksp, R=R, lams=LAMBDAS, seed=SEED, max_iter=MAX_ITER)
F.fig_lambda_images(sweep, save_dir=SAVE)
F.fig_lambda_curve(sweep, save_dir=SAVE)
plt.show()

## 10. Robustez ao ruído (Figura `exp6_noise`)

Junta ruído gaussiano complexo ao espaço-$k$ (σ relativo ao máximo de $|k|$) antes da subamostragem —
aproxima melhor uma aquisição real do que o caso sem ruído.

**Para mudar a figura:** `R` e `SIGMAS`.

In [ ]:
# >>> PARÂMETROS
R      = 4
SIGMAS = (0, 1e-4, 5e-4, 1e-3, 2e-3)

nsweep = F.noise_sweep(img, ksp, R=R, sigmas=SIGMAS, lamda=LAMBDA, seed=SEED, max_iter=MAX_ITER)
F.fig_noise(nsweep, save_dir=SAVE)
plt.show()

## 11. Gerar tudo de uma vez (opcional)

O mesmo que o notebook, mas pela linha de comandos. Útil para regenerar as figuras do relatório.
```bash
python run_demo.py                              # todas as figuras + tabela
python run_demo.py --only masktype --R 8        # só a figura exp3 para R=8
python run_demo.py --only lambda --lams 1e-3 3e-3 1e-2 3e-2 1e-1
python run_demo.py --list                       # lista de figuras e opções
```

## 12. Interpretação e limitações

- **PSNR:** quanto maior, menor o erro em relação à referência.
- **SSIM:** mais perto de 1 = maior semelhança estrutural.
- **R:** maior aceleração → menos dados → reconstrução mais difícil.
- **Máscara:** o CS só funciona bem com amostragem incoerente (aleatória); com a regular o artefacto é estruturado e a regularização wavelet não o remove.
- **Limitações:** imagem MNI152 já reconstruída (magnitude) para simular o espaço-$k$ (*inverse crime*), uma só bobina ideal,
  sem dados brutos de scanner nem doentes. Os resultados mostram as funcionalidades do SigPy, não validam desempenho clínico.